# 02 — Feature Engineering

**Purpose:** construct leakage-safe lifecycle snapshots from the canonical daily dataset. Future outcomes are targets only; current scoring snapshots contain no `target_*` columns.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))


In [2]:
from game_performance.config import SETTINGS
from game_performance.features import build_all_historical_snapshots, categorical_features, model_features
from game_performance.io import load_dataframe, save_artifact

df = load_dataframe(SETTINGS.processed_dir / "clean_game_daily.pkl")


In [3]:
historical_snapshots = build_all_historical_snapshots(df, SETTINGS.horizon_strategy)
summary = {key: len(frame) for key, frame in historical_snapshots.items()}
summary


{(7, 7): 9803,
 (7, 14): 9609,
 (7, 30): 9532,
 (14, 7): 8783,
 (14, 14): 9301,
 (14, 30): 9142,
 (14, 60): 8301,
 (21, 14): 8607,
 (21, 30): 8674,
 (21, 60): 7876}

In [4]:
example_key = next(iter(historical_snapshots))
example = historical_snapshots[example_key]
print("Example configuration:", example_key)
print("Categorical features:", categorical_features(example))
print("Model feature count:", len(model_features(example)))
example.head()


Example configuration: (7, 7)
Categorical features: ['SystemName', 'TournamentName', 'Launch_Month', 'Launch_Weekday']
Model feature count: 100


,Game_ID,SystemName,TournamentName,First_Observed_Date,Launch_Month,Launch_Weekday,Obs_Window,Horizon,Lifecycle_Stage,active_observation_days,...,days_since_ggr_peak,obs_ggr_pct,bets_per_client,stake_per_client,ggr_per_client,target_clients,target_betcount,target_stakes,target_ggr,target_ggr_pct
0,AmusnetSlots::AmusnetS-100 Bulky Fruits,AmusnetSlots,AmusnetS-100 Bulky Fruits,2026-02-01,February,Sunday,7,7,Day 7,7,...,5.0,0.032268,118.649466,390.788256,12.610142,423,48979,175348.0,6688.90,0.038146
1,AmusnetSlots::AmusnetS-100 Cats,AmusnetSlots,AmusnetS-100 Cats,2026-02-01,February,Sunday,7,7,Day 7,7,...,5.0,0.045818,69.019514,155.452900,7.122550,21910,1701258,4405087.0,139108.20,0.031579
2,AmusnetSlots::AmusnetS-100 Power Hot,AmusnetSlots,AmusnetS-100 Power Hot,2026-02-01,February,Sunday,7,7,Day 7,7,...,5.0,0.003174,90.402986,184.388237,0.585321,22933,2291184,5152790.0,195424.23,0.037926
3,AmusnetSlots::AmusnetS-2 Dragons,AmusnetSlots,AmusnetS-2 Dragons,2026-02-01,February,Sunday,7,7,Day 7,7,...,1.0,0.037937,73.837925,125.011345,4.742553,728,62629,109315.8,3545.20,0.032431
4,AmusnetSlots::AmusnetS-20 Bulky Fruits,AmusnetSlots,AmusnetS-20 Bulky Fruits,2026-02-01,February,Sunday,7,7,Day 7,7,...,3.0,0.018874,79.487831,156.282359,2.949600,8267,715709,1299967.0,43801.95,0.033695


In [5]:
save_artifact(historical_snapshots, SETTINGS.feature_dir / "historical_snapshots.joblib")
print("Saved feature snapshots")


Saved feature snapshots
